# Tissue oxygenation s$_t$O$_2$ inference using Monte Carlo-simulated and surrogate model-generated reflectances

Note: To ensure data access and correct path resolution, run this notebook with a kernel that uses the internal version of the `htc` package.

In [ ]:
import os
import pickle
import sys
import warnings
from collections.abc import Callable, Iterator
from dataclasses import dataclass
from pathlib import Path

import matplotlib as mpl
import matplotlib.legend as mlegend
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from htc.tivita.rgb import hsi_to_rgb
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D

repo_root = Path(__name__).resolve().parents[3]
sys.path.insert(0, str(repo_root))

from mcmlnet.experiments.data_loaders.config import DataConfig
from mcmlnet.experiments.data_loaders.porcine_clamping_data import (
    ClampingDataLoader,
)
from mcmlnet.experiments.data_loaders.simulation import (
    GenericSimulationLoader,
    SimulationDataLoaderManager,
)
from mcmlnet.experiments.data_loaders.utils import subsample_data
from mcmlnet.experiments.plotting import setup_plot_style
from mcmlnet.experiments.utils import (
    fit_knn,
    predict_knn,
)
from mcmlnet.training.data_loading.preprocessing import PreProcessor

load_dotenv()
setup_plot_style()

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
if not os.path.exists(os.environ["plot_save_dir"]):
    os.makedirs(os.environ["plot_save_dir"])

## Load simulated dataset and real dataloader

In [ ]:
@dataclass
class DatasetSplit:
    """Container for feature/target pairs."""

    features: torch.Tensor
    targets: torch.Tensor


@dataclass
class DatasetSplits:
    """Train/val/test splits for a dataset."""

    train: DatasetSplit
    val: DatasetSplit
    test: DatasetSplit

    def iter_splits(self) -> Iterator[tuple[str, DatasetSplit]]:
        """Yield named splits to simplify for-loops in callers."""
        yield from (
            ("train", self.train),
            ("val", self.val),
            ("test", self.test),
        )


def split_dataset(
    features: torch.Tensor,
    targets: torch.Tensor,
    preprocessor: PreProcessor,
) -> DatasetSplits:
    """Split tensors into deterministic train/val/test partitions.

    Args:
        features: Feature tensor of shape (n_samples, n_features).
        targets: Target tensor aligned with `features`.
        preprocessor: PreProcessor providing consistent split indices.

    Returns:
        DatasetSplits containing `DatasetSplit` instances for each partition.
    """
    train_idx = preprocessor.consistent_data_split_ids(features, mode="train")
    val_idx = preprocessor.consistent_data_split_ids(features, mode="val")
    test_idx = preprocessor.consistent_data_split_ids(features, mode="test")

    return DatasetSplits(
        train=DatasetSplit(features[train_idx], targets[train_idx]),
        val=DatasetSplit(features[val_idx], targets[val_idx]),
        test=DatasetSplit(features[test_idx], targets[test_idx]),
    )


def load_params_and_reflectance_data() -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """Load paired physiological parameters and reflectance tensors.

    Returns:
        Tuple containing:
            physiological_params: DataFrame with physiological targets (e.g., SaO2).
            adapted_sim_reflectances: Monte Carlo simulated reflectances (np.ndarray).
            adapted_surr_reflectances: Surrogate-model reflectances corresponding to
                the simulations.
    """
    # Extract corresponding Monte Carlo-simulated data pairs
    adapted_sim_reflectances = GenericSimulationLoader().load_reflectances()
    preprocessor = PreProcessor(
        val_percent=DataConfig.DEFAULT_VAL_PERCENT,
        test_percent=DataConfig.DEFAULT_TEST_PERCENT,
    )
    train_ids = preprocessor.consistent_data_split_ids(
        adapted_sim_reflectances, mode="train"
    )
    adapted_sim_reflectances = adapted_sim_reflectances[train_ids]

    # Load underlying physiological parameters
    physiological_params = GenericSimulationLoader().load_physiological_parameters()

    # Subsample to the 100k saved surrogate model samples
    physiological_params_ids = subsample_data(
        np.arange(len(physiological_params)),
        DataConfig.DEFAULT_SUBSAMPLE_SIZE_SURROGATE_MODEL,
    )
    physiological_params = physiological_params.iloc[physiological_params_ids]
    adapted_sim_reflectances = subsample_data(
        adapted_sim_reflectances, DataConfig.DEFAULT_SUBSAMPLE_SIZE_SURROGATE_MODEL
    )

    # Load camera adapted surrogate reflectances
    adapted_surr_reflectances = (
        SimulationDataLoaderManager()._load_adapted_generic_data().numpy()
    )

    return physiological_params, adapted_sim_reflectances, adapted_surr_reflectances

In [ ]:
# Load simulated data
physiological_params, adapted_sim_reflectances, adapted_surr_reflectances = (
    load_params_and_reflectance_data()
)

# grab all SaO2 (simulation) columns as a (n_samples, n_layers) frame
sao2 = physiological_params.xs("SaO2", axis=1, level=1)

# Compute common data splits
mc_splits = split_dataset(
    torch.from_numpy(adapted_sim_reflectances),
    torch.from_numpy(sao2.to_numpy()[:, [0]]),
    PreProcessor(
        val_percent=DataConfig.DEFAULT_VAL_PERCENT,
        test_percent=DataConfig.DEFAULT_TEST_PERCENT,
    ),
)
surr_splits = split_dataset(
    torch.from_numpy(adapted_surr_reflectances),
    torch.from_numpy(sao2.to_numpy()[:, [0]]),
    PreProcessor(
        val_percent=DataConfig.DEFAULT_VAL_PERCENT,
        test_percent=DataConfig.DEFAULT_TEST_PERCENT,
    ),
)

In [ ]:
# Define validation dataloader
data_loader = ClampingDataLoader()
dataloader_val, metadata_val = data_loader.load_validation_data()

# Print some dataloader properties
for batch in dataloader_val:
    for key, val in batch.items():
        print(key, val, type(val))
    break
len(dataloader_val.dataset.paths)

In [ ]:
# Define test dataloader
dataloader_test, metadata_test = data_loader.load_test_data()

# Print some dataloader properties
for batch in dataloader_test:
    for key, val in batch.items():
        print(key, val, type(val))
    break
len(dataloader_test.dataset.paths)

In [ ]:
# Load label mapping
label_mapping_aortic = {
    "background": 0,
    "colon": 1,
    "fat (subcutaneous)": 2,
    "fat (visceral)": 3,
    "gallbladder": 4,
    "liver": 5,
    "muscle": 6,
    "peritoneum": 7,
    "skin": 8,
    "small bowel": 9,
    "spleen": 10,
    "stomach": 11,
    "unclear_organic": 12,
    "overlap": 254,
    "unlabeled": 255,
}

## Visualize data and labels

In [ ]:
# Assign tab20 cmap colors also used in the PCA analysis
tab20 = mpl.cm.get_cmap("tab20")
labels = [
    "colon",
    "omentum",
    "small bowel",
    "fat (subcutaneous)",
    "gallbladder",
    "liver",
    "skin",
    "stomach",
    "fat (visceral)",
    "peritoneum",
    "spleen",
    "muscle",
    "pancreas",
    "kidney",
    "diaphragm",
    "major vein",
    "heart",
    "lung",
    "cauterized tissue",
    "hepatic ligament",
    "esophagus",
    "artery",
    "bladder",
]
LABEL_COLORS = {
    label: mpl.colors.to_hex(tab20(i))  # or tab20(i / tab20.N) for floats
    for i, label in enumerate(labels)
}

labels = list(LABEL_COLORS)
cmap = ListedColormap([LABEL_COLORS[k] for k in labels])
norm = BoundaryNorm(range(len(labels) + 1), cmap.N)

In [ ]:
cube = batch["features"][0].cpu().numpy()
rgb = hsi_to_rgb(cube.astype(np.float32))
ann = batch["labels"][0].cpu().numpy()

# pick a colormap (or build your own RGBA) and an alpha mask
labels = np.unique(ann)
cmap = mpl.cm.get_cmap("tab20", len(labels))
norm = mpl.colors.BoundaryNorm(np.r_[labels, labels[-1] + 1], cmap.N)
ticks = labels
tick_names = [name for name, idx in label_mapping_aortic.items() if idx in labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(ann, cmap=cmap, norm=norm)
axes[0].set_title("Labels")
axes[0].axis("off")

axes[1].imshow(rgb)
# axes[1].imshow(ann, cmap=cmap, norm=norm, alpha=0.1)
axes[1].set_title("HSI → RGB")
axes[1].axis("off")

sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(
    sm, ax=axes, ticks=ticks, location="right", fraction=0.035, pad=0.02
)
cbar.set_ticklabels(tick_names)

plt.show()

## Compute sO2 using kNN lookup

In [ ]:
# Define bootstrapping function
def torch_bootstrap_ci(
    data: torch.Tensor,
    confidence_level: float = 0.95,
    n_resamples: int = 1_000,
    metric: Callable = torch.mean,
    device: torch.device | str = "cuda",
) -> tuple[float, float]:
    """
    Torch-based bootstrapping (using resampling) computation
    of CI's for specified metric function.

    Args:
        data: Tensor of sO2 values.
        confidence_level: The confidence level of the CI.
        n_resamples: Amount of resamples during bootstrapping.
        metric: Callable function to compute the CI for.
        device: Device to use for torch computation.

    Raises:
        ValueError: If input data shape is invalid.

    Returns:
        tuple[float, float], computed lower and upper CI bounds
    """
    n = data.shape[0]
    if n == 0:
        raise ValueError("data must be non-empty!")
    if data.ndim > 1:
        raise ValueError("data must be 1D!")

    # Draw resample indices with replacement
    with torch.no_grad():
        idx = torch.randint(0, n, (n_resamples, n), device=device)
        samples = data.to(device)[idx]
        metrics = metric(samples, dim=1)

        if isinstance(metrics, tuple):
            metrics = metrics.values
        metrics = metrics.cpu()

    lower = torch.quantile(metrics, (1 - confidence_level) / 2)
    upper = torch.quantile(metrics, 1 - (1 - confidence_level) / 2)
    return lower.item(), upper.item()

In [ ]:
# Compute organ-specific sO2 based on nearest neighbours
K_NEIGHBOURS = 1
L1_NORMALIZED = False
USE_MAE_THRESHOLDING = True  # as in spectral recall analysis


def compute_subject_specific_data_knn(
    subject_name: str,
    cutoff: int = 100,
    k: int = K_NEIGHBOURS,
    sim_split: DatasetSplits = mc_splits,
    dataloader: torch.utils.data.DataLoader = dataloader_val,
    metadata_df: pd.DataFrame = metadata_val,
) -> pd.DataFrame:
    """
    Compute DataFrame containing median and IQR sO2 of a specific subject
    based on the nearest simulation neighbours.
    """
    # hard-coded, for now (could use metadata lookup)
    assert subject_name in ["P190", "P191", "P192", "P193"]
    if subject_name in ["P192", "P193"]:
        warnings.warn("Using test subjects, make sure this is intended!", stacklevel=2)
    # Create subject specific dataloader
    matching_idx = [
        i
        for i, path in enumerate(dataloader.dataset.paths)
        if subject_name in path.image_name()
    ]

    subject_subset = torch.utils.data.Subset(dataloader.dataset, matching_idx)
    subject_subset_loader = torch.utils.data.DataLoader(
        subject_subset, batch_size=1, num_workers=0, shuffle=False
    )

    rows = {
        "image_name": [],
        "subject_name": [],
        "phase_type": [],
        "timestamp": [],
        "rel_time": [],
        "label_id": [],
        "median_sto2": [],
        "mean_sto2": [],
        "lower_quartile": [],
        "upper_quartile": [],
        "ci_low": [],
        "ci_high": [],
    }

    # Collect simulation data
    sim_reflectance = sim_split.train.features[:, :cutoff]
    sim_so2 = sim_split.train.targets
    start_time = None

    for _i, sample in enumerate(subject_subset_loader):
        # Collect real kidney spectra
        features = sample["features"].squeeze()
        labels = sample["labels"].squeeze()
        image_name = sample["image_name"][0]

        # Collect metadata
        metadata_row = metadata_df.loc[metadata_df.image_name == image_name]
        meta_dict = metadata_row.squeeze().to_dict()
        phase_type = meta_dict["phase_type"]
        timestamp = pd.to_datetime(meta_dict["timestamp"])
        # Initialize start and thus clamping time
        if not start_time:
            start_time = timestamp

        # Iterate over organ classes
        for label_id in labels.unique():
            mask = labels == label_id
            if not mask.any():
                continue
            real_pixels = features[mask][:, :cutoff]

            if L1_NORMALIZED:
                l1_sims = sim_reflectance.sum(dim=1, keepdim=True)
                l1_real = real_pixels.sum(dim=1, keepdim=True)

                # Fit kNN on simulation data
                knn = fit_knn(sim_reflectance / l1_sims, k=k)

                # Predict nearest simulation
                ids, _ = predict_knn(knn, real_pixels / l1_real)
            else:
                # Fit kNN on simulation data
                knn = fit_knn(sim_reflectance, k=k)

                # Predict nearest simulation
                ids, _ = predict_knn(knn, real_pixels)

            if USE_MAE_THRESHOLDING:
                # MAE thresholding using the spectral recall threshold
                mae_threshold = 0.02
                maes = torch.mean(
                    torch.abs(sim_reflectance[ids] - real_pixels.unsqueeze(1)), dim=-1
                ).flatten()
                mae_mask = maes < mae_threshold
                ids = ids.flatten()[mae_mask]
            else:
                ids = ids.flatten()

            # Collect "assigned" sO2s
            so2s = sim_so2[ids].numpy()

            # Compute median and quartiles
            median = np.median(so2s)
            mean = np.mean(so2s)
            q1 = np.percentile(so2s, 25)
            q3 = np.percentile(so2s, 75)
            ci_low, ci_high = torch_bootstrap_ci(
                sim_so2[ids].squeeze(), n_resamples=500, metric=torch.mean
            )

            # Append metadata row
            rows["image_name"].append(image_name)
            rows["subject_name"].append(image_name[:4])
            rows["phase_type"].append(phase_type)
            rows["timestamp"].append(timestamp)
            rows["rel_time"].append((timestamp - start_time) / np.timedelta64(1, "s"))
            rows["label_id"].append(int(label_id))
            rows["median_sto2"].append(median)
            rows["mean_sto2"].append(mean)
            rows["lower_quartile"].append(q1)
            rows["upper_quartile"].append(q3)
            rows["ci_low"].append(ci_low)
            rows["ci_high"].append(ci_high)

    return pd.DataFrame(rows)

In [ ]:
# Define functions for plotting the results
ABS_TO_PERCENT = 100


def plot_sto2_time_progression(
    df: pd.DataFrame,
    excluded_label_names: list[str],
    ax: plt.Axes,
    show_hline: bool = True,
    marker: str = "o",
    linestyle: str = "-",
    title: str = r"Mean s$_t$O$_2$ per organ with 95% CI of the mean",
) -> None:
    """
    Plot oxygenation traces over time, including vertical lines,
    highlighting the time points of (un-)clamping.
    """
    # Invert and apply label mapping
    label_mapping_inv = {idx: name for name, idx in label_mapping_aortic.items()}
    df["label_name"] = df["label_id"].map(label_mapping_inv)

    sec_per_minute = 60

    # Exclude organs to not show
    for label_name, group in df.groupby("label_name"):
        if label_name in excluded_label_names:
            continue
        color = LABEL_COLORS.get(label_name, "#999999")
        group["rel_time"] /= sec_per_minute

        ax.plot(
            group["rel_time"],
            group["mean_sto2"] * ABS_TO_PERCENT,
            marker=marker,
            color=color,
            linestyle=linestyle,
            label=label_name,
            markersize=3,
        )
        ax.fill_between(
            group["rel_time"],
            group["ci_low"] * ABS_TO_PERCENT,  # or lower / upper_quantile
            group["ci_high"] * ABS_TO_PERCENT,
            alpha=0.2,
            color=color,
        )
        baseline = group.iloc[0]["mean_sto2"] * ABS_TO_PERCENT
        if show_hline:
            ax.hlines(
                y=baseline,
                xmin=group["rel_time"].min(),
                xmax=group["rel_time"].max(),
                colors=color,
                linestyles="--",
                linewidth=1,
            )

    # Add phase information as vertical lines + annotations
    ymin, ymax = 0, ABS_TO_PERCENT

    # Get timestamp
    phase_starts = df.groupby("phase_type")["rel_time"].head(1)
    phase_starts /= sec_per_minute
    first_clamped = phase_starts.values[1] - 0.5
    first_reperfused = phase_starts.values[2] - 0.5

    # Plot vertical lines and annotations
    ax.axvline(first_clamped, color="k", lw=1.5, zorder=0)
    ax.text(
        first_clamped,
        0.02,
        "clamp applied",
        rotation=90,
        va="bottom",
        ha="right",
    )
    ax.axvline(first_reperfused, color="k", lw=1.5, zorder=0)
    ax.text(
        first_reperfused - 2,
        0.57,
        "clamp released",
        rotation=90,
        va="bottom",
        ha="left",
    )

    ax.set_ylim(ymin, ymax)

    ax.set_xlabel("Minutes after Clamping")
    ax.set_ylabel(r"s$_t$O$_2$ [%]")
    ax.set_title(title)
    ax.legend(title="Organ Label", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(True)


def plot_sto2_time_progression_minimal(
    df: pd.DataFrame,
    excluded_label_names: list[str],
    ax: plt.Axes,
    show_hline: bool = True,
    marker: str = "o",
    markersize: float = 3.0,
    linestyle: str = "-",
    line_alpha: float = 0.8,
    title: str = r"Mean s$_t$O$_2$ per organ with 95% CI of the mean",
) -> None:
    """Plot only oxygenation traces over time."""
    # Invert and apply label mapping
    label_mapping_inv = {idx: name for name, idx in label_mapping_aortic.items()}
    df["label_name"] = df["label_id"].map(label_mapping_inv)

    sec_per_minute = 60

    # Exclude organs to not show
    for label_name, group in df.groupby("label_name"):
        if label_name in excluded_label_names:
            continue
        color = LABEL_COLORS.get(label_name, "#999999")
        group["rel_time"] /= sec_per_minute

        ax.plot(
            group["rel_time"],
            group["mean_sto2"] * ABS_TO_PERCENT,
            marker=marker,
            color=color,
            linestyle=linestyle,
            label=label_name,
            markersize=markersize,
            alpha=line_alpha,
        )
        ax.fill_between(
            group["rel_time"],
            group["ci_low"] * ABS_TO_PERCENT,  # or lower / upper_quantile
            group["ci_high"] * ABS_TO_PERCENT,
            alpha=0.2,
            color=color,
        )
        baseline = group.iloc[0]["mean_sto2"] * ABS_TO_PERCENT
        if show_hline:
            ax.hlines(
                y=baseline,
                xmin=group["rel_time"].min(),
                xmax=group["rel_time"].max(),
                colors=color,
                linestyles="--",
                linewidth=1,
            )
    ax.set_xlabel("Minutes after Clamping")
    ax.set_ylabel(r"s$_t$O$_2$ [%]")
    ax.set_ylim(0, ABS_TO_PERCENT)
    ax.set_title(title)
    ax.legend(title="Organ Label", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(True)

In [ ]:
N_LABELS_VISCERAL = [6, 6, 7, 5]
N_LABELS_PERIPHERAL = [3, 4, 4, 4]


def _clone_handle(handle: Line2D) -> Line2D:
    """Return a detached copy so Matplotlib can format it independently."""
    new = Line2D([], [])
    new.update_from(handle)
    return new


def manually_set_legend(
    single_ax: plt.Axes, n_labels: int, label_cutoff_multiple_pigs: bool = False
) -> mlegend.Legend:
    """Manipulate labels (manually set subheadings) of a single subplot."""
    handles, labels = single_ax.get_legend_handles_labels()
    groups = {
        "Monte Carlo-based": (handles[:n_labels], labels[:n_labels]),
        "Surrogate-based": (handles[n_labels:], labels[n_labels:]),
    }
    if label_cutoff_multiple_pigs:
        handles_surr, labels_surr = groups["Surrogate-based"]
        groups["Surrogate-based"] = (handles_surr[:n_labels], labels_surr[:n_labels])

    legend_handles, legend_labels = [], []

    for group_title, (grp_handles, grp_labels) in groups.items():
        legend_handles.append(Line2D([], [], linestyle="none", linewidth=0))
        legend_labels.append(group_title)
        legend_handles.extend(_clone_handle(h) for h in grp_handles)
        legend_labels.extend(grp_labels)

    leg = single_ax.legend(
        legend_handles,
        legend_labels,
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        handlelength=2,
        handletextpad=1,
        borderpad=1,
    )
    for txt, _h in zip(leg.get_texts(), legend_handles, strict=False):
        if txt.get_text() in ["Monte Carlo-based", "Surrogate-based"]:
            # txt.set_fontweight("bold")
            txt.set_fontsize("medium")

    return leg

### 5NN

In [ ]:
# Collect validation subject data
surrogate_dataframes = []
mc_dataframes = []
for _subjects in ["P190", "P191"]:
    surrogate_dataframes.append(
        compute_subject_specific_data_knn(_subjects, sim_split=surr_splits)
    )
    mc_dataframes.append(
        compute_subject_specific_data_knn(_subjects, sim_split=mc_splits)
    )

In [ ]:
# Collect test subject data
for _subjects in ["P192", "P193"]:
    surrogate_dataframes.append(
        compute_subject_specific_data_knn(
            _subjects,
            sim_split=surr_splits,
            dataloader=dataloader_test,
            metadata_df=metadata_test,
        )
    )
    mc_dataframes.append(
        compute_subject_specific_data_knn(
            _subjects,
            sim_split=mc_splits,
            dataloader=dataloader_test,
            metadata_df=metadata_test,
        )
    )

#### Cache and load 5NN data

In [ ]:
cache_dir = Path(os.environ["cache_dir"])
cache_path = cache_dir / "subject_5nn_cache.pkl"

In [ ]:
# Cache dataframes for later/ quicker reloading and plot redesign
dataframes_5nn = {
    "surrogate_dataframes": surrogate_dataframes,
    "mc_dataframes": mc_dataframes,
}

with cache_path.open("wb") as fh:
    pickle.dump(dataframes_5nn, fh)

In [ ]:
# Reload cached data
with cache_path.open("rb") as fh:
    cache = pickle.load(fh)
surrogate_dataframes = cache["surrogate_dataframes"]
mc_dataframes = cache["mc_dataframes"]

#### Plot results

In [ ]:
# Try overlapping multiple pigs with Lenas suggestions (...?)
fig, ax = plt.subplots(figsize=(6.5, 3))

size_map = {"P191": 5, "P192": 3, "P193": 1}
rejected_labels = [
    "unclear_organic",
    "fat (subcutaneous)",
    "peritoneum",
    "stomach",
    "fat (visceral)",
    "background",
    "gallbladder",
    "muscle",
]

for i, surrogate_df in enumerate(surrogate_dataframes):
    if i < 1:
        continue
    _markersize = size_map[surrogate_df.subject_name.iloc[0]]
    plot_sto2_time_progression_minimal(
        mc_dataframes[i],
        rejected_labels,
        ax=ax,
        markersize=_markersize,
        line_alpha=0.5,
    )
    plot_sto2_time_progression_minimal(
        surrogate_df,
        rejected_labels,
        ax=ax,
        show_hline=False,
        marker="d",
        markersize=_markersize,
        linestyle=":",
        title=(
            r"Mean s$_t$O$_2$ per organ with 95% CI of the mean; P191, P192, and P193"
        ),
        line_alpha=0.5,
    )

# Compute min-max bands across subjects using the row IDs as proxy
mc_all = pd.concat(mc_dataframes, ignore_index=True).sort_values(
    ["label_name", "subject_name", "rel_time"]
)
mc_all["sample_idx"] = mc_all.groupby(["label_name", "subject_name"]).cumcount()
envelope = (
    mc_all.groupby(["label_name", "sample_idx"])["mean_sto2"]
    .agg(sto2_min="min", sto2_max="max", sto2_mean="mean")
    .reset_index()
)
for organ, band in envelope.groupby("label_name"):
    if organ in rejected_labels:
        continue
    ax.fill_between(
        band["sample_idx"],
        band["sto2_min"] * ABS_TO_PERCENT,
        band["sto2_max"] * ABS_TO_PERCENT,
        color=LABEL_COLORS.get(organ, "#999999"),
        alpha=0.25,
    )


# Manually define phase information as axvlines
ymin, ymax = 0, 1

# Get timestamps of clamping
phase_starts_p191 = mc_dataframes[1].groupby("phase_type")["rel_time"].head(1)
phase_starts_p191 /= 60
first_reperfused_p191 = phase_starts_p191.values[2] - 0.5

phase_starts_p193 = mc_dataframes[3].groupby("phase_type")["rel_time"].head(1)
phase_starts_p193 /= 60
first_clamped_p193 = phase_starts_p193.values[1] - 0.25
first_reperfused_p193 = phase_starts_p193.values[2] - 0.5

# Plot vertical lines and annotations
ax.axvline(first_clamped_p193, color="k", lw=1.5, zorder=0)
ax.text(
    first_clamped_p193,
    0.02,
    "clamp applied",
    rotation=90,
    va="bottom",
    ha="right",
)
ax.axvline(first_reperfused_p191, color="k", lw=1.5, zorder=0)
ax.axvline(first_reperfused_p193, color="k", lw=1.5, zorder=0)
ax.text(
    first_reperfused_p193 - 1.5,
    0.53,
    "clamp(s) released",
    rotation=90,
    va="bottom",
    ha="left",
)

legend_organs = manually_set_legend(ax, 5, label_cutoff_multiple_pigs=True)

# Manually add size legend for subject encoding
size_handles = [
    Line2D(
        [],
        [],
        marker="o",
        linestyle="",
        markersize=size_map[name],
        color="gray",
        label=name,
    )
    for name in ["P191", "P192", "P193"]
]
legend_sizes = ax.legend(
    handles=size_handles, title="Subject", bbox_to_anchor=(1.02, 0), loc="lower left"
)
ax.add_artist(legend_organs)


plt.tight_layout()
plt.show()

In [ ]:
# 1x2 base results of all pigs
for i, surrogate_df in enumerate(surrogate_dataframes):
    subject_name = surrogate_df.subject_name.unique()[0]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    plot_sto2_time_progression(
        mc_dataframes[i],
        [
            "unclear_organic",
            "skin",
            "fat (subcutaneous)",
            "peritoneum",
            "background",
            "muscle",
        ],
        ax=ax[0],
    )
    plot_sto2_time_progression(
        surrogate_df,
        [
            "unclear_organic",
            "skin",
            "fat (subcutaneous)",
            "peritoneum",
            "background",
            "muscle",
        ],
        ax=ax[0],
        show_hline=False,
        marker="d",
        linestyle=":",
        title=rf"Median $\pm$ IQR s$_t$O$_2$ per organ, {subject_name}, visceral",
    )
    manually_set_legend(ax[0], N_LABELS_VISCERAL[i])
    plot_sto2_time_progression(
        surrogate_df,
        [
            "unclear_organic",
            "colon",
            "stomach",
            "small bowel",
            "spleen",
            "stomach",
            "liver",
            "fat (visceral)",
            "background",
            "gallbladder",
        ],
        ax=ax[1],
    )
    plot_sto2_time_progression(
        mc_dataframes[i],
        [
            "unclear_organic",
            "colon",
            "stomach",
            "small bowel",
            "spleen",
            "stomach",
            "liver",
            "fat (visceral)",
            "background",
            "gallbladder",
        ],
        ax=ax[1],
        show_hline=False,
        marker="d",
        linestyle=":",
        title=rf"Median $\pm$ IQR s$_t$O$_2$ per organ, {subject_name}, peripheral",
    )
    manually_set_legend(ax[1], N_LABELS_PERIPHERAL[i])
    plt.tight_layout()
    plt.show()

### 1NN

In [ ]:
# Collect validation subject data
surrogate_dataframes_1nn = []
mc_dataframes_1nn = []
for _subjects in ["P190", "P191"]:
    surrogate_dataframes_1nn.append(
        compute_subject_specific_data_knn(_subjects, sim_split=surr_splits, k=1)
    )
    mc_dataframes_1nn.append(
        compute_subject_specific_data_knn(_subjects, sim_split=mc_splits, k=1)
    )

In [ ]:
# Collect test subject data
for _subjects in ["P192", "P193"]:
    surrogate_dataframes_1nn.append(
        compute_subject_specific_data_knn(
            _subjects,
            k=1,
            sim_split=surr_splits,
            dataloader=dataloader_test,
            metadata_df=metadata_test,
        )
    )
    mc_dataframes_1nn.append(
        compute_subject_specific_data_knn(
            _subjects,
            k=1,
            sim_split=mc_splits,
            dataloader=dataloader_test,
            metadata_df=metadata_test,
        )
    )

#### Cache and load 1NN data

In [ ]:
# Cache dataframes for later/ quicker reloading and plot redesign
cache_dir = Path(os.environ["cache_dir"])
cache_path = cache_dir / "subject_1nn_cache.pkl"

In [ ]:
dataframes_1nn = {
    "surrogate_dataframes": surrogate_dataframes_1nn,
    "mc_dataframes": mc_dataframes_1nn,
}

with cache_path.open("wb") as fh:
    pickle.dump(dataframes_1nn, fh)

In [ ]:
# Reload cached data
with cache_path.open("rb") as fh:
    cache = pickle.load(fh)
surrogate_dataframes_1nn = cache["surrogate_dataframes"]
mc_dataframes_1nn = cache["mc_dataframes"]

#### Plot results

In [ ]:
# Try overlapping multiple pigs with Lenas suggestions (...?)
fig, ax = plt.subplots(figsize=(10, 5))

for i, surrogate_df in enumerate(surrogate_dataframes_1nn):
    if i < 1:
        continue
    plot_sto2_time_progression(
        mc_dataframes_1nn[i],
        [
            "unclear_organic",
            "fat (subcutaneous)",
            "peritoneum",
            "stomach",
            "fat (visceral)",
            "background",
            "gallbladder",
            "muscle",
        ],
        ax=ax,
    )
    plot_sto2_time_progression(
        surrogate_df,
        [
            "unclear_organic",
            "fat (subcutaneous)",
            "peritoneum",
            "stomach",
            "fat (visceral)",
            "background",
            "gallbladder",
            "muscle",
        ],
        ax=ax,
        show_hline=False,
        marker="d",
        linestyle=":",
        title=r"Median $\pm$ IQR s$_t$O$_2$ per organ, P190 & P191",
    )

manually_set_legend(ax, 5, label_cutoff_multiple_pigs=True)
plt.show()

In [ ]:
# Try overlapping multiple pigs with Lenas suggestions (...?)
fig, ax = plt.subplots(figsize=(6.5, 3))

size_map = {"P191": 3, "P192": 3, "P193": 3}
linestyle_map = {"P191": "-", "P192": ":", "P193": "--"}
rejected_labels = [
    "unclear_organic",
    "fat (subcutaneous)",
    "peritoneum",
    "stomach",
    "fat (visceral)",
    "background",
    "gallbladder",
    "muscle",
]

for i, surrogate_df in enumerate(surrogate_dataframes_1nn):
    if i < 1:
        continue
    _markersize = size_map[surrogate_df.subject_name.iloc[0]]
    _linestype = linestyle_map[surrogate_df.subject_name.iloc[0]]
    plot_sto2_time_progression_minimal(
        mc_dataframes_1nn[i],
        rejected_labels,
        ax=ax,
        show_hline=False,
        markersize=_markersize,
        linestyle=_linestype,
    )
    plot_sto2_time_progression_minimal(
        surrogate_df,
        rejected_labels,
        ax=ax,
        show_hline=False,
        marker="d",
        markersize=_markersize,
        linestyle="",
        title=(
            r"Mean s$_t$O$_2$ per organ with 95% CI of the mean; P191, P192, and P193"
        ),
    )

# Compute min-max bands across subjects by binning the time (x-axis)
bin_size = 60  # seconds
mc_all = pd.concat(mc_dataframes_1nn, ignore_index=True)
mc_all["bin_min"] = (mc_all["rel_time"] / bin_size).round().astype(int)
mc_all["time_min"] = mc_all["bin_min"] * bin_size / 60
envelope = (
    mc_all.groupby(["label_name", "time_min"])["mean_sto2"]
    .agg(sto2_min="min", sto2_max="max", sto2_mean="mean")
    .reset_index()
)

for organ, band in envelope.groupby("label_name"):
    if organ in rejected_labels:
        continue
    ax.fill_between(
        band["time_min"],
        band["sto2_min"] * ABS_TO_PERCENT,
        band["sto2_max"] * ABS_TO_PERCENT,
        color=LABEL_COLORS.get(organ, "#999999"),
        alpha=0.25,
        edgecolor="none",
    )


# Manually define phase information as axvlines
# Get timestamps of clamping
phase_starts_p191 = mc_dataframes_1nn[1].groupby("phase_type")["rel_time"].head(1)
phase_starts_p191 /= 60
first_reperfused_p191 = phase_starts_p191.values[2] - 0.5

phase_starts_p193 = mc_dataframes_1nn[3].groupby("phase_type")["rel_time"].head(1)
phase_starts_p193 /= 60
first_clamped_p193 = phase_starts_p193.values[1] - 0.25
first_reperfused_p193 = phase_starts_p193.values[2] - 0.5

# Plot vertical lines and annotations
ax.axvline(first_clamped_p193, color="k", lw=1.5, zorder=0)
ax.text(
    first_clamped_p193,
    0.02,
    "clamp applied",
    rotation=90,
    va="bottom",
    ha="right",
)
ax.axvline(first_reperfused_p191, color="k", lw=1.5, zorder=0)
ax.text(
    first_reperfused_p193 + 0.8,
    0.71,
    "P191, P192",
    rotation=90,
    va="bottom",
    ha="left",
)

ax.axvline(first_reperfused_p193, color="k", lw=1.5, zorder=0)
ax.text(
    first_reperfused_p193,
    0.63,
    "clamp released\nP193",
    rotation=90,
    va="bottom",
    ha="right",
)

legend_organs = manually_set_legend(ax, 5, label_cutoff_multiple_pigs=True)

# Manually add size legend for subject encoding
size_handles = [
    Line2D(
        [],
        [],
        marker="o",
        linestyle=linestyle_map[name],
        markersize=size_map[name],
        color="gray",
        label=name,
    )
    for name in ["P191", "P192", "P193"]
]
legend_sizes = ax.legend(
    handles=size_handles, title="Subject", bbox_to_anchor=(1.02, 0), loc="lower left"
)
ax.add_artist(legend_organs)

plt.xlim(-1, 40)
plt.savefig(
    os.path.join(
        os.environ["plot_save_dir"],
        "oxygenation_1nn.svg",
    )
)
plt.show()

In [ ]:
# Try overlapping multiple pigs with Lenas suggestions (...?)
fig, ax = plt.subplots(figsize=(6.5, 3))

size_map = {"P191": 3, "P192": 3, "P193": 3}
marker_map = {"P191": "o", "P192": "P", "P193": "d"}
linestyle_map = {"P191": "-", "P192": ":", "P193": "--"}
rejected_labels = [
    "unclear_organic",
    "fat (subcutaneous)",
    "peritoneum",
    "stomach",
    "fat (visceral)",
    "background",
    "gallbladder",
    "muscle",
]

for i, surrogate_df in enumerate(surrogate_dataframes_1nn):
    if i < 1:
        continue
    _markersize = size_map[surrogate_df.subject_name.iloc[0]]
    _marker = marker_map[surrogate_df.subject_name.iloc[0]]
    _linestype = linestyle_map[surrogate_df.subject_name.iloc[0]]
    plot_sto2_time_progression_minimal(
        surrogate_df,
        rejected_labels,
        ax=ax,
        show_hline=False,
        marker=_marker,
        markersize=_markersize,
        linestyle=_linestype,
        title=(
            r"Mean s$_t$O$_2$ per organ with 95% CI of the mean; P191, P192, and P193"
        ),
    )

# Compute min-max bands across subjects by binning the time (x-axis)
bin_size = 60  # seconds
df_all = pd.concat(surrogate_dataframes_1nn, ignore_index=True)
df_all["bin_min"] = (df_all["rel_time"] / bin_size).round().astype(int)
df_all["time_min"] = df_all["bin_min"] * bin_size / 60
envelope = (
    df_all.groupby(["label_name", "time_min"])["mean_sto2"]
    .agg(sto2_min="min", sto2_max="max", sto2_mean="mean")
    .reset_index()
)

for organ, band in envelope.groupby("label_name"):
    if organ in rejected_labels:
        continue
    ax.fill_between(
        band["time_min"],
        band["sto2_min"] * ABS_TO_PERCENT,
        band["sto2_max"] * ABS_TO_PERCENT,
        color=LABEL_COLORS.get(organ, "#999999"),
        alpha=0.25,
        edgecolor="none",
    )


# Manually define phase information as axvlines
ymin, ymax = 0, 1

# Get timestamps of clamping
phase_starts_p191 = (
    surrogate_dataframes_1nn[1].groupby("phase_type")["rel_time"].head(1)
)
phase_starts_p191 /= 60
first_reperfused_p191 = phase_starts_p191.values[2] - 0.5

phase_starts_p193 = (
    surrogate_dataframes_1nn[3].groupby("phase_type")["rel_time"].head(1)
)
phase_starts_p193 /= 60
first_clamped_p193 = phase_starts_p193.values[1] - 0.25
first_reperfused_p193 = phase_starts_p193.values[2] - 0.5

# Plot vertical lines and annotations
ax.axvline(first_clamped_p193, color="k", lw=1.5, zorder=0)
ax.text(
    first_clamped_p193,
    0.02,
    "clamp applied",
    rotation=90,
    va="bottom",
    ha="right",
)
ax.axvline(first_reperfused_p191, color="k", lw=1.5, zorder=0)
ax.text(
    first_reperfused_p193 + 0.8,
    0.71,
    "P191, P192",
    rotation=90,
    va="bottom",
    ha="left",
)

ax.axvline(first_reperfused_p193, color="k", lw=1.5, zorder=0)
ax.text(
    first_reperfused_p193,
    0.63,
    "clamp released\nP193",
    rotation=90,
    va="bottom",
    ha="right",
)

# Manually add legend for subject encoding
handles, labels = ax.get_legend_handles_labels()
n_labels = 5

grouped_handles, grouped_labels = [], []
for title, slc in [("Organ", slice(0, n_labels))]:
    grouped_handles.append(Line2D([], [], linestyle="none", linewidth=0))
    grouped_labels.append(title)
    grouped_handles.extend(_clone_handle(h) for h in handles[slc])
    grouped_labels.extend(labels[slc])

size_handles = [
    Line2D(
        [],
        [],
        marker=marker_map[name],
        linestyle=linestyle_map[name],
        markersize=size_map[name],
        color="gray",
        label=name,
    )
    for name in ["P191", "P192", "P193"]
]
grouped_handles.append(Line2D([], [], linestyle="none", linewidth=0))
grouped_labels.append("Subject")
grouped_handles.extend(size_handles)
grouped_labels.extend([h.get_label() for h in size_handles])

legend = ax.legend(
    grouped_handles,
    grouped_labels,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    handlelength=2,
    handletextpad=1,
)


plt.xlim(-1, 40)
plt.savefig(
    os.path.join(
        os.environ["plot_save_dir"],
        "oxygenation_1nn.svg",
    )
)
plt.show()

In [ ]:
# Create organ-specific panels
panel_organs = [
    (r"Mean s$_t$O$_2$ in Liver", ["liver"]),
    (r"Mean s$_t$O$_2$ in Colon and Small Bowel", ["colon", "small bowel"]),
    (r"Mean s$_t$O$_2$ in Spleen", ["spleen"]),
    (r"Mean s$_t$O$_2$ in Skin", ["skin"]),
]

fig, axes = plt.subplots(2, 2, figsize=(6.5, 4), sharex=True, sharey=True)
axes = axes.ravel()

size_map = {"P191": 3, "P192": 3, "P193": 3}
marker_map = {"P191": "o", "P192": "P", "P193": "d"}
linestyle_map = {"P191": "-", "P192": ":", "P193": "--"}
subjects = ["P191", "P192", "P193"]

line_alpha = 0.8
fill_alpha = 0.4
sec_per_minute = 60

# Envelope shading per organ - assuming that envelope is already computed
for ax, (title, organs) in zip(axes, panel_organs, strict=False):
    for organ in organs:
        band = envelope[envelope["label_name"] == organ]
        ax.fill_between(
            band["time_min"],
            band["sto2_min"] * ABS_TO_PERCENT,
            band["sto2_max"] * ABS_TO_PERCENT,
            color=LABEL_COLORS.get(organ, "#999999"),
            alpha=fill_alpha,
            edgecolor="none",
        )

    for subj in subjects:
        surrogate_df = next(
            df for df in surrogate_dataframes_1nn if df.subject_name.iloc[0] == subj
        )
        surrogate_f = surrogate_df[surrogate_df["label_name"].isin(organs)].copy()

        # Invert and apply label mapping
        label_mapping_inv = {idx: name for name, idx in label_mapping_aortic.items()}
        surrogate_f["label_name"] = surrogate_f["label_id"].map(label_mapping_inv)

        # Plot by organ
        for label_name, group in surrogate_f.groupby("label_name"):
            color = LABEL_COLORS.get(label_name, "#999999")
            group["rel_time"] /= sec_per_minute

            ax.plot(
                group["rel_time"],
                group["mean_sto2"] * ABS_TO_PERCENT,
                marker=marker_map[subj],
                color=color,
                linestyle=linestyle_map[subj],
                label=label_name,
                markersize=size_map[subj],
                alpha=line_alpha,
            )

    # Add axis labels and titles
    if "spleen" in title.lower() or "skin" in title.lower():
        ax.set_xlabel("Minutes after Clamping")
    if "liver" in title.lower() or "spleen" in title.lower():
        ax.set_ylabel(r"s$_t$O$_2$ [%]")
    ax.set_ylim(0, ABS_TO_PERCENT)
    ax.set_xlim(-3, 40)
    ax.set_title(title)
    ax.grid(True)

    # Manually define phase information as axvlines on every panel
    # Get timestamps of clamping
    phase_starts_p191 = mc_dataframes_1nn[1].groupby("phase_type")["rel_time"].head(1)
    phase_starts_p191 /= 60
    first_reperfused_p191 = phase_starts_p191.values[2] - 0.5

    phase_starts_p193 = mc_dataframes_1nn[3].groupby("phase_type")["rel_time"].head(1)
    phase_starts_p193 /= 60
    first_clamped_p193 = phase_starts_p193.values[1] - 0.25
    first_reperfused_p193 = phase_starts_p193.values[2] - 0.5

    # Plot vertical lines and annotations
    ax.axvline(first_clamped_p193, color="k", lw=1.5, zorder=0)
    ax.text(
        first_clamped_p193,
        3,
        "clamp applied",
        rotation=90,
        va="bottom",
        ha="right",
    )
    ax.axvline(first_reperfused_p191, color="k", lw=1.5, zorder=0)
    ax.text(
        first_reperfused_p191 + 0.5,
        3 if "colon" in title.lower() else 51,
        "P191, P192",
        rotation=90,
        va="bottom",
        ha="left",
    )

    ax.axvline(first_reperfused_p193, color="k", lw=1.5, zorder=0)
    ax.text(
        first_reperfused_p193,
        65,
        "clamp\nreleased\nP193",
        rotation=90,
        va="bottom",
        ha="right",
    )

# Manually add legend for subject encoding
handles, labels = ax.get_legend_handles_labels()
grouped_handles, grouped_labels = [], []


# Organ section
organ_order = ["colon", "liver", "skin", "small bowel", "spleen"]
grouped_handles.append(Line2D([], [], linestyle="none", linewidth=0))
grouped_labels.append("Organ")
organ_handles = [
    Line2D(
        [],
        [],
        marker="o",
        linestyle="-",
        linewidth=1.5,
        color=LABEL_COLORS[o],
        markersize=5,
        label=o,
    )
    for o in organ_order
]
grouped_handles.extend(organ_handles)
grouped_labels.extend([h.get_label() for h in organ_handles])

# Subject section
size_handles = [
    Line2D(
        [],
        [],
        marker=marker_map[name],
        linestyle=linestyle_map[name],
        markersize=size_map[name],
        color="gray",
        label=name,
    )
    for name in ["P191", "P192", "P193"]
]
grouped_handles.append(Line2D([], [], linestyle="none", linewidth=0))
grouped_labels.append("Subject")
grouped_handles.extend(size_handles)
grouped_labels.extend([h.get_label() for h in size_handles])

legend = fig.legend(
    grouped_handles,
    grouped_labels,
    loc="center left",
    bbox_to_anchor=(0.98, 0.5),
    handlelength=2,
    handletextpad=1,
)

fig.tight_layout()
plt.savefig(
    os.path.join(
        os.environ["plot_save_dir"],
        "oxygenation_1nn_panels.svg",
    )
)
plt.show()

In [ ]:
# Create organ-specific panels - color-code subjects instead of organs
panel_organs = [
    (r"Mean s$_t$O$_2$ in Liver", ["liver"]),
    (r"Mean s$_t$O$_2$ in Colon", ["colon"]),
    (r"Mean s$_t$O$_2$ in Spleen", ["spleen"]),
    (r"Mean s$_t$O$_2$ in Skin", ["skin"]),
]

fig, axes = plt.subplots(2, 2, figsize=(6.5, 3.5), sharex=True, sharey=True)
axes = axes.ravel()

subject_colors = {"P191": "#1f77b4", "P192": "#2ca02c", "P193": "#d62728"}
size_map = {"P191": 3, "P192": 3, "P193": 3}
marker_map = {"P191": "o", "P192": "o", "P193": "o"}
linestyle_map = {"P191": "-", "P192": ":", "P193": "--"}
subjects = ["P191", "P192", "P193"]

line_alpha = 1.0
fill_alpha = 1.0
sec_per_minute = 60

# Envelope shading per organ - assuming that envelope is already computed
for ax, (title, organs) in zip(axes, panel_organs, strict=False):
    for subj in subjects:
        surrogate_df = next(
            df for df in surrogate_dataframes_1nn if df.subject_name.iloc[0] == subj
        )
        surrogate_f = surrogate_df[surrogate_df["label_name"].isin(organs)].copy()

        # Invert and apply label mapping
        label_mapping_inv = {idx: name for name, idx in label_mapping_aortic.items()}
        surrogate_f["label_name"] = surrogate_f["label_id"].map(label_mapping_inv)

        # Plot by organ
        for label_name, group in surrogate_f.groupby("label_name"):
            color = LABEL_COLORS.get(label_name, "#999999")
            group["rel_time"] /= sec_per_minute

            ax.plot(
                group["rel_time"],
                group["mean_sto2"] * ABS_TO_PERCENT,
                marker=marker_map[subj],
                color=subject_colors[subj],
                linestyle=linestyle_map[subj],
                label=label_name,
                markersize=size_map[subj],
                alpha=line_alpha,
            )

    # Add axis labels and titles
    if "spleen" in title.lower() or "skin" in title.lower():
        ax.set_xlabel("Minutes after Clamping")
    if "liver" in title.lower() or "spleen" in title.lower():
        ax.set_ylabel(r"s$_t$O$_2$ [%]")
    ax.set_ylim(0, ABS_TO_PERCENT)
    ax.set_xlim(-3, 40)
    ax.set_title(title)
    ax.grid(True)

    # Manually define phase information as axvlines on every panel
    # Get timestamps of clamping
    phase_starts_p191 = mc_dataframes_1nn[1].groupby("phase_type")["rel_time"].head(1)
    phase_starts_p191 /= 60
    first_reperfused_p191 = phase_starts_p191.values[2] - 0.5

    phase_starts_p193 = mc_dataframes_1nn[3].groupby("phase_type")["rel_time"].head(1)
    phase_starts_p193 /= 60
    first_clamped_p193 = phase_starts_p193.values[1] - 0.25
    first_reperfused_p193 = phase_starts_p193.values[2] - 0.5

    # Plot vertical lines and annotations
    ax.axvline(first_clamped_p193, color="k", lw=1.5, zorder=0)
    ax.text(
        first_clamped_p193,
        3,
        "clamp applied",
        rotation=90,
        va="bottom",
        ha="right",
    )
    ax.axvline(first_reperfused_p191, color="k", lw=1.5, zorder=0)
    ax.text(
        first_reperfused_p191 + 0.5,
        3 if "colon" in title.lower() else 43,
        "P191, P192",
        rotation=90,
        va="bottom",
        ha="left",
    )

    ax.axvline(first_reperfused_p193, color="k", lw=1.5, zorder=0)
    ax.text(
        first_reperfused_p193,
        60,
        "clamp\nreleased\nP193",
        rotation=90,
        va="bottom",
        ha="right",
    )

# Manually add legend for subject encoding
handles, labels = ax.get_legend_handles_labels()
grouped_handles, grouped_labels = [], []
grouped_handles.append(Line2D([], [], linestyle="none", linewidth=0))
grouped_labels.append("Subject")
size_handles = [
    Line2D(
        [],
        [],
        marker=marker_map[name],
        linestyle=linestyle_map[name],
        markersize=size_map[name],
        color=subject_colors[name],
        label=name,
    )
    for name in subjects
]
grouped_handles.extend(size_handles)
grouped_labels.extend([h.get_label() for h in size_handles])

legend = fig.legend(
    grouped_handles,
    grouped_labels,
    loc="center left",
    bbox_to_anchor=(0.98, 0.5),
    handlelength=2,
    handletextpad=1,
)

fig.tight_layout()
plt.savefig(
    os.path.join(
        os.environ["plot_save_dir"],
        "oxygenation_1nn_panels_simple.svg",
    )
)
plt.show()

In [ ]:
# 1x2 base results of all pigs
for i, surrogate_df in enumerate(surrogate_dataframes_1nn):
    subject_name = surrogate_df.subject_name.unique()[0]
    fig, ax = plt.subplots(1, 2, figsize=(6.5, 1.5))
    plot_sto2_time_progression(
        mc_dataframes_1nn[i],
        [
            "unclear_organic",
            "skin",
            "fat (subcutaneous)",
            "peritoneum",
            "background",
            "muscle",
        ],
        ax=ax[0],
    )
    plot_sto2_time_progression(
        surrogate_df,
        [
            "unclear_organic",
            "skin",
            "fat (subcutaneous)",
            "peritoneum",
            "background",
            "muscle",
        ],
        ax=ax[0],
        show_hline=False,
        marker="d",
        linestyle=":",
        title=rf"Mean s$_t$O$_2$ per organ, {subject_name}, visceral",
    )
    manually_set_legend(ax[0], N_LABELS_VISCERAL[i])
    plot_sto2_time_progression(
        surrogate_df,
        [
            "unclear_organic",
            "colon",
            "stomach",
            "small bowel",
            "spleen",
            "stomach",
            "liver",
            "fat (visceral)",
            "background",
            "gallbladder",
        ],
        ax=ax[1],
    )
    plot_sto2_time_progression(
        mc_dataframes_1nn[i],
        [
            "unclear_organic",
            "colon",
            "stomach",
            "small bowel",
            "spleen",
            "stomach",
            "liver",
            "fat (visceral)",
            "background",
            "gallbladder",
        ],
        ax=ax[1],
        show_hline=False,
        marker="d",
        linestyle=":",
        title=rf"Mean s$_t$O$_2$ per organ, {subject_name}, peripheral",
    )
    manually_set_legend(ax[1], N_LABELS_PERIPHERAL[i])
    plt.savefig(
        os.path.join(
            os.environ["plot_save_dir"],
            f"oxygenation_1nn_{subject_name}.svg",
        )
    )
    plt.show()